# 用于 CSV 文件的简单 RAG（检索增强生成）系统

## 概述

此代码实现了用于处理和查询 CSV 文档的基本检索增强生成 (RAG) 系统。系统将文档内容编码到支持存储中，然后可以查询该存储来检索相关信息。

# CSV 文件结构和用例
CSV 文件包含虚拟客户数据，包括名字、姓氏、公司等各种属性。该数据集将用于 RAG 用例，有助于创建客户信息问答系统。

## 关键组件

1. 加载并分割csv文件。
2. 使用 [FAISS](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/) 和 OpenAI 嵌入支持存储创建
3. 查询处理后的文档的查询引擎设置
4. 针对 csv 数据创建问题和答案。

## 方法详细信息

### 文档预处理

1. 使用LlamaIndex的[PagedCSVReader](https://docs.llamaindex.ai/en/stable/api_reference/readers/file/#llama_index.readers.file.PagedCSVReader)加载csv
2. 该读取器将每一行以及表的相应列名称转换为 LlamaIndex 文档。不再应用进一步的分割。


### 支持存储创建

1. OpenAI 嵌入用于创建文本块的向量表示。
2. 根据这些嵌入创建 FAISS 存储存储，以实现高效的相似性搜索。

### 查询引擎设置

1. 查询引擎被配置为获取与给定查询最相关的块，然后回答问题。

## 这种方法的好处

1. 可扩展性：可以通过分块处理大型文档。
2. 灵活性：易于调整块大小和检索结果数量等参数。
3. 高效：利用FAISS在高维空间中进行快速相似性搜索。
4. 与高级 NLP 集成：使用 OpenAI 嵌入来实现最先进的文本表示。

## 结论

这个简单的 RAG 系统为构建更复杂的信息检索和问答系统提供了坚实的基础。通过将文档内容编码为可搜索的支持存储，它可以有效地检索相关信息以响应查询。此方法对于需要快速访问 CSV 文件中的特定信息的应用程序特别有用。

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install faiss-cpu llama-index pandas python-dotenv

In [1]:
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.readers.file import PagedCSVReader
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core import VectorStoreIndex
import faiss
import os
import pandas as pd
from dotenv import load_dotenv


# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')


# Llamaindex global settings for llm and embeddings
EMBED_DIMENSION=512
Settings.llm = OpenAI(model="gpt-3.5-turbo")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=EMBED_DIMENSION)

### CSV 文件结构和用例
CSV 文件包含虚拟客户数据，包括名字、姓氏、公司等各种属性。该数据集将用于 RAG 用例，有助于创建客户信息问答系统。

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/customers-100.csv https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv


In [ ]:
file_path = ('data/customers-100.csv') # insert the path of the csv file
data = pd.read_csv(file_path)

# Preview the csv file
data.head()

### 支撑存储

In [3]:
# Create FaisVectorStore to store embeddings
fais_index = faiss.IndexFlatL2(EMBED_DIMENSION)
vector_store = FaissVectorStore(faiss_index=fais_index)

### 将 CSV 数据作为文档加载和处理

In [4]:
csv_reader = PagedCSVReader()

reader = SimpleDirectoryReader( 
    input_files=[file_path],
    file_extractor= {".csv": csv_reader}
    )

docs = reader.load_data()

In [5]:
# Check a sample chunk
print(docs[0].text)

Index: 1
Customer Id: DD37Cf93aecA6Dc
First Name: Sheryl
Last Name: Baxter
Company: Rasmussen Group
City: East Leonard
Country: Chile
Phone 1: 229.077.5154
Phone 2: 397.884.0519x718
Email: zunigavanessa@smith.info
Subscription Date: 2020-08-24
Website: http://www.stephenson.com/


### 摄取管道

In [6]:
pipeline = IngestionPipeline(
    vector_store=vector_store,
    documents=docs
)

nodes = pipeline.run()

### 创建查询引擎

In [7]:
vector_store_index = VectorStoreIndex(nodes)
query_engine = vector_store_index.as_query_engine(similarity_top_k=2)

### 使用基于 CSV 数据的问题查询 rag 机器人

In [8]:
response = query_engine.query("which company does sheryl Baxter work for?")
response.response

'Rasmussen Group'

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--simple-csv-rag-with-llamaindex)